# Network Intrusion Detection Pipeline

This notebook implements a complete machine learning pipeline for network intrusion detection using Logistic Regression and Decision Tree models.

## Import Required Libraries

In [37]:
import importlib
import sys
import pandas as pd

# Remove cached modules to force reload
if 'src.preprocessing' in sys.modules:
    del sys.modules['src.preprocessing']

from src.preprocessing import load_and_preprocess_data
from src.models import train_logistic_regression, train_decision_tree
from src.evaluation import evaluate_model

## Load and Preprocess Data

In [38]:
print("Starting Data Preprocessing Pipeline")

dataset_path = "archive/KDDTrain+.txt"

X_train, X_test, y_train, y_test = load_and_preprocess_data(dataset_path)

print("\nPreprocessing Completed Successfully!")
print(f"X_train shape (scaled matrices): {X_train.shape}")
print(f"X_test shape (scaled matrices): {X_test.shape}")
print("\nTarget label distribution (Training Set):")
print(y_train.value_counts())

Starting Data Preprocessing Pipeline

Preprocessing Completed Successfully!
X_train shape (scaled matrices): (100778, 119)
X_test shape (scaled matrices): (25195, 119)

Target label distribution (Training Set):
label
0    53921
1    46857
Name: count, dtype: int64


## Feature Selection / Dimensionality Reduction

In this section, we will reduce the number of features from 110+ to 20 most important features using two methods:
1. **SelectKBest** - Uses ANOVA F-value scoring
2. **Random Forest Feature Importance** - Identifies features based on tree-based importance

We'll compare if using fewer features affects training speed and model accuracy.

In [39]:
# Reload preprocessing module to ensure feature selection functions are available
import importlib
if 'src.preprocessing' in sys.modules:
    importlib.reload(sys.modules['src.preprocessing'])

from src.preprocessing import select_features_kbest, select_features_random_forest
import time

# Feature Selection using SelectKBest 
k_features = 20
start_time = time.time()

X_train_selected_kbest, X_test_selected_kbest, selector_kbest, kbest_scores = select_features_kbest(
    X_train, X_test, y_train, k=k_features
)

kbest_time = time.time() - start_time
print(f"SelectKBest execution time: {kbest_time:.4f} seconds")


FEATURE SELECTION - SelectKBest (ANOVA F-value)
Original number of features: 119
Selected number of features: 20
Dimensionality reduction: 83.19%

Top 20 Most Important Features:
 Feature_Index         Score  Selected
           117 135178.657501      True
            25 131789.550276      True
            29 110155.606238      True
            30  94218.602023      True
             8  91566.584787      True
            35  75967.039456      True
            34  74846.406121      True
            21  74326.736761      True
           113  74064.526338      True
            22  73331.281362      True
            19  50295.486545      True
            63  46548.006620      True
            88  25440.101877      True
            28  16705.630019      True
            51   7260.256085      True
            24   6921.263461      True
            37   6918.371183      True
            23   6896.825661      True
            36   6840.002479      True
            31   6391.181048      True



c:\Users\fland\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [16] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\fland\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [40]:
# Feature Selection using Random Forest Feature Importance
start_time = time.time()

X_train_selected_rf, X_test_selected_rf, rf_indices, rf_scores = select_features_random_forest(
    X_train, X_test, y_train, k=k_features
)

rf_time = time.time() - start_time
print(f"Random Forest Feature Selection execution time: {rf_time:.4f} seconds")


FEATURE SELECTION - Random Forest Importance
Original number of features: 119
Selected number of features: 20
Dimensionality reduction: 83.19%

Top 20 Most Important Features:
 Feature_Index  Importance  Rank
             1    0.107818     1
             2    0.098078     2
           117    0.070062     3
            29    0.056987     4
            30    0.054369     5
            34    0.048048     6
             8    0.045231     7
            26    0.040415     8
            31    0.038624     9
            19    0.037486    10
            25    0.037250    11
            32    0.035738    12
            35    0.027182    13
           113    0.023441    14
            63    0.022862    15
            21    0.022328    16
            22    0.022132    17
            33    0.020557    18
            36    0.018754    19
            28    0.017867    20

Random Forest Feature Selection execution time: 1.3469 seconds


In [41]:
# Compare model performance with different feature sets
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

print("\n" + "="*80)
print("COMPARISON: Model Performance with Different Feature Sets")
print("="*80)

# Models with ORIGINAL features
print("\n1. ORIGINAL FEATURES (110+ columns):")
lr_original = LogisticRegression(max_iter=1000, random_state=42)
lr_original.fit(X_train, y_train)
lr_acc_original = accuracy_score(y_test, lr_original.predict(X_test))
lr_f1_original = f1_score(y_test, lr_original.predict(X_test))

dt_original = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_original.fit(X_train, y_train)
dt_acc_original = accuracy_score(y_test, dt_original.predict(X_test))
dt_f1_original = f1_score(y_test, dt_original.predict(X_test))

print(f"  Logistic Regression  - Accuracy: {lr_acc_original*100:.2f}%, F1-Score: {lr_f1_original*100:.2f}%")
print(f"  Decision Tree        - Accuracy: {dt_acc_original*100:.2f}%, F1-Score: {dt_f1_original*100:.2f}%")

# Models with SELECTKBEST features (20 columns)
print("\n2. SELECTKBEST FEATURES (20 columns - 82% reduction):")
lr_kbest = LogisticRegression(max_iter=1000, random_state=42)
lr_kbest.fit(X_train_selected_kbest, y_train)
lr_acc_kbest = accuracy_score(y_test, lr_kbest.predict(X_test_selected_kbest))
lr_f1_kbest = f1_score(y_test, lr_kbest.predict(X_test_selected_kbest))

dt_kbest = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_kbest.fit(X_train_selected_kbest, y_train)
dt_acc_kbest = accuracy_score(y_test, dt_kbest.predict(X_test_selected_kbest))
dt_f1_kbest = f1_score(y_test, dt_kbest.predict(X_test_selected_kbest))

print(f"  Logistic Regression  - Accuracy: {lr_acc_kbest*100:.2f}%, F1-Score: {lr_f1_kbest*100:.2f}%")
print(f"  Decision Tree        - Accuracy: {dt_acc_kbest*100:.2f}%, F1-Score: {dt_f1_kbest*100:.2f}%")

# Models with RANDOM FOREST FEATURES (20 columns)
print("\n3. RANDOM FOREST FEATURES (20 columns - 82% reduction):")
lr_rf = LogisticRegression(max_iter=1000, random_state=42)
lr_rf.fit(X_train_selected_rf, y_train)
lr_acc_rf = accuracy_score(y_test, lr_rf.predict(X_test_selected_rf))
lr_f1_rf = f1_score(y_test, lr_rf.predict(X_test_selected_rf))

dt_rf = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_rf.fit(X_train_selected_rf, y_train)
dt_acc_rf = accuracy_score(y_test, dt_rf.predict(X_test_selected_rf))
dt_f1_rf = f1_score(y_test, dt_rf.predict(X_test_selected_rf))

print(f"  Logistic Regression  - Accuracy: {lr_acc_rf*100:.2f}%, F1-Score: {lr_f1_rf*100:.2f}%")
print(f"  Decision Tree        - Accuracy: {dt_acc_rf*100:.2f}%, F1-Score: {dt_f1_rf*100:.2f}%")

# Summary comparison table
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
comparison_data = {
    'Feature Set': ['Original (110+)', 'SelectKBest (20)', 'Random Forest (20)'],
    'LR Accuracy': [f"{lr_acc_original*100:.2f}%", f"{lr_acc_kbest*100:.2f}%", f"{lr_acc_rf*100:.2f}%"],
    'LR F1-Score': [f"{lr_f1_original*100:.2f}%", f"{lr_f1_kbest*100:.2f}%", f"{lr_f1_rf*100:.2f}%"],
    'DT Accuracy': [f"{dt_acc_original*100:.2f}%", f"{dt_acc_kbest*100:.2f}%", f"{dt_acc_rf*100:.2f}%"],
    'DT F1-Score': [f"{dt_f1_original*100:.2f}%", f"{dt_f1_kbest*100:.2f}%", f"{dt_f1_rf*100:.2f}%"]
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("="*80)


COMPARISON: Model Performance with Different Feature Sets

1. ORIGINAL FEATURES (110+ columns):
  Logistic Regression  - Accuracy: 97.14%, F1-Score: 96.91%
  Decision Tree        - Accuracy: 99.66%, F1-Score: 99.63%

2. SELECTKBEST FEATURES (20 columns - 82% reduction):
  Logistic Regression  - Accuracy: 94.69%, F1-Score: 94.25%
  Decision Tree        - Accuracy: 98.69%, F1-Score: 98.60%

3. RANDOM FOREST FEATURES (20 columns - 82% reduction):
  Logistic Regression  - Accuracy: 93.74%, F1-Score: 93.22%
  Decision Tree        - Accuracy: 99.61%, F1-Score: 99.58%

SUMMARY TABLE
       Feature Set LR Accuracy LR F1-Score DT Accuracy DT F1-Score
   Original (110+)      97.14%      96.91%      99.66%      99.63%
  SelectKBest (20)      94.69%      94.25%      98.69%      98.60%
Random Forest (20)      93.74%      93.22%      99.61%      99.58%


## Extended Feature Engineering - Testing Multiple K Values

To find the optimal number of features, we will test different k values and compare model performance. This helps us determine the best trade-off between model complexity and accuracy.

In [ ]:
# Reload preprocessing module
import importlib
if 'src.preprocessing' in sys.modules:
    importlib.reload(sys.modules['src.preprocessing'])

from src.preprocessing import test_multiple_k_values, create_interaction_features, create_domain_specific_features

k_values = [10, 15, 20, 25, 30]
results_df = test_multiple_k_values(X_train, X_test, y_train, y_test, k_values=k_values)


TESTING MULTIPLE K VALUES FOR FEATURE SELECTION



c:\Users\fland\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [16] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\fland\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


K=10 ( 91.6% reduction)
  LR: Acc= 90.20%, F1= 89.21%
  DT: Acc= 94.89%, F1= 94.41%



c:\Users\fland\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [16] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\fland\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


K=15 ( 87.4% reduction)
  LR: Acc= 94.56%, F1= 94.12%
  DT: Acc= 98.15%, F1= 98.01%



...

K=20 ( 83.2% reduction)
  LR: Acc= 94.69%, F1= 94.25%
  DT: Acc= 98.69%, F1= 98.60%



c:\Users\fland\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [16] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\fland\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


K=25 ( 79.0% reduction)
  LR: Acc= 96.13%, F1= 95.79%
  DT: Acc= 98.56%, F1= 98.45%



c:\Users\fland\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [16] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\fland\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


K=30 ( 74.8% reduction)
  LR: Acc= 96.41%, F1= 96.11%
  DT: Acc= 98.96%, F1= 98.89%


SUMMARY TABLE:
 K Reduction LR_Accuracy  LR_F1 DT_Accuracy  DT_F1 Avg_Accuracy Avg_F1
10     91.6%      90.20% 89.21%      94.89% 94.41%       92.55% 91.81%
15     87.4%      94.56% 94.12%      98.15% 98.01%       96.35% 96.06%
20     83.2%      94.69% 94.25%      98.69% 98.60%       96.69% 96.43%
25     79.0%      96.13% 95.79%      98.56% 98.45%       97.34% 97.12%
30     74.8%      96.41% 96.11%      98.96% 98.89%       97.69% 97.50%



## Feature Engineering - Creating Interaction Features

Interaction features combine the information from multiple features. For network intrusion detection, interactions between error rates, connection counts, and other metrics can reveal attack patterns not visible in individual features.

In [ ]:
X_train_inter, X_test_inter = create_interaction_features(X_train_selected_kbest, X_test_selected_kbest)

lr_inter = LogisticRegression(max_iter=1000, random_state=42)
lr_inter.fit(X_train_inter, y_train)
lr_acc_inter = accuracy_score(y_test, lr_inter.predict(X_test_inter))
lr_f1_inter = f1_score(y_test, lr_inter.predict(X_test_inter))

dt_inter = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_inter.fit(X_train_inter, y_train)
dt_acc_inter = accuracy_score(y_test, dt_inter.predict(X_test_inter))
dt_f1_inter = f1_score(y_test, dt_inter.predict(X_test_inter))

print(f"Interaction Features: LR={lr_acc_inter*100:.2f}%, DT={dt_acc_inter*100:.2f}%")


CREATING INTERACTION FEATURES
Original features: 20
Creating 10 interaction features
Total features after interactions: 30


Testing models with interaction features:
Logistic Regression - Accuracy: 94.82%, F1: 94.39%
Decision Tree        - Accuracy: 98.86%, F1: 98.79%


## Feature Engineering - Creating Domain-Specific Features

Domain-specific features are engineered based on expert knowledge about network intrusion detection. These features include:
- **Total Connection Anomaly Score**: Sum of error rates and suspicious indicators
- **Connection Concentration Metric**: Variance of feature values showing traffic diversity
- **Attack Indicator**: Maximum suspicious pattern value indicating potential attacks

In [ ]:
X_train_domain, X_test_domain = create_domain_specific_features(X_train, X_test, y_train, y_test)

lr_domain = LogisticRegression(max_iter=1000, random_state=42)
lr_domain.fit(X_train_domain, y_train)
lr_acc_domain = accuracy_score(y_test, lr_domain.predict(X_test_domain))
lr_f1_domain = f1_score(y_test, lr_domain.predict(X_test_domain))

dt_domain = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_domain.fit(X_train_domain, y_train)
dt_acc_domain = accuracy_score(y_test, dt_domain.predict(X_test_domain))
dt_f1_domain = f1_score(y_test, dt_domain.predict(X_test_domain))

print(f"Domain-Specific Features: LR={lr_acc_domain*100:.2f}%, DT={dt_acc_domain*100:.2f}%")


CREATING DOMAIN-SPECIFIC FEATURES
Original features: 119
Domain-specific features added: 3
  1. Total Connection Anomaly Score
  2. Connection Concentration Metric
  3. Attack Indicator (Suspicious Pattern)
Total features: 122


Testing models with domain-specific features:
Logistic Regression - Accuracy: 97.13%, F1: 96.91%
Decision Tree        - Accuracy: 99.65%, F1: 99.63%


## Feature Engineering Comparison Summary

Compare all feature engineering approaches to determine which provides the best performance for our classifiers.

In [54]:
all_features_comparison = {
    'Feature Set': ['Original (115+)', 'SelectKBest (20)', 'Random Forest (20)', 'Interaction', 'Domain-Specific'],
    'Features': [X_train.shape[1], X_train_selected_kbest.shape[1], X_train_selected_rf.shape[1], X_train_inter.shape[1], X_train_domain.shape[1]],
    'LR Accuracy': [f"{lr_acc_original*100:.2f}%", f"{lr_acc_kbest*100:.2f}%", f"{lr_acc_rf*100:.2f}%", f"{lr_acc_inter*100:.2f}%", f"{lr_acc_domain*100:.2f}%"],
    'DT Accuracy': [f"{dt_acc_original*100:.2f}%", f"{dt_acc_kbest*100:.2f}%", f"{dt_acc_rf*100:.2f}%", f"{dt_acc_inter*100:.2f}%", f"{dt_acc_domain*100:.2f}%"],
}

comparison_all_df = pd.DataFrame(all_features_comparison)
print("\nFeature Engineering Comparison:")
print(comparison_all_df.to_string(index=False))
print(f"\nBest Balance: SelectKBest (k=20) - {100 * (1 - 20/X_train.shape[1]):.1f}% reduction with {dt_acc_kbest*100:.2f}% DT accuracy")


Feature Engineering Comparison:
       Feature Set  Features LR Accuracy DT Accuracy
   Original (115+)       119      97.14%      99.66%
  SelectKBest (20)        20      94.69%      98.69%
Random Forest (20)        20      93.74%      99.61%
       Interaction        30      94.82%      98.86%
   Domain-Specific       122      97.13%      99.65%

Best Balance: SelectKBest (k=20) - 83.2% reduction with 98.69% DT accuracy


## EXPERIMENT: All Features vs Selected Features Comparison

This section compares model performance using all 115+ original features versus the top 20 selected features after dimensionality reduction via SelectKBest. This experiment demonstrates the trade-off between model complexity and accuracy.

In [53]:
print("\n" + "="*80)
print("EXPERIMENT: All Features (115+) vs Selected Features (Top 20)")
print("="*80 + "\n")

all_vs_selected = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'All Features': [f"{lr_acc_original*100:.2f}%", f"{dt_acc_original*100:.2f}%"],
    'Top 20': [f"{lr_acc_kbest*100:.2f}%", f"{dt_acc_kbest*100:.2f}%"],
    'Loss': [f"{(lr_acc_kbest - lr_acc_original)*100:+.2f}%", f"{(dt_acc_kbest - dt_acc_original)*100:+.2f}%"],
    'Reduction': [f"{100 * (1 - 20/X_train.shape[1]):.1f}%", f"{100 * (1 - 20/X_train.shape[1]):.1f}%"]
})

print(all_vs_selected.to_string(index=False))
print("\nConclusion: SelectKBest (k=20) RECOMMENDED - minimal accuracy loss (<2.5%) with 83% reduction")


EXPERIMENT: All Features (115+) vs Selected Features (Top 20)

              Model All Features Top 20   Loss Reduction
Logistic Regression       97.14% 94.69% -2.44%     83.2%
      Decision Tree       99.66% 98.69% -0.96%     83.2%

Conclusion: SelectKBest (k=20) RECOMMENDED - minimal accuracy loss (<2.5%) with 83% reduction


In [52]:
import time

print("\nTraining Time Comparison:")

start = time.time()
lr_all = LogisticRegression(max_iter=1000, random_state=42)
lr_all.fit(X_train, y_train)
lr_all_time = time.time() - start

start = time.time()
dt_all = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_all.fit(X_train, y_train)
dt_all_time = time.time() - start

start = time.time()
lr_sel = LogisticRegression(max_iter=1000, random_state=42)
lr_sel.fit(X_train_selected_kbest, y_train)
lr_sel_time = time.time() - start

start = time.time()
dt_sel = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_sel.fit(X_train_selected_kbest, y_train)
dt_sel_time = time.time() - start

timing_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'All Features (s)': [f"{lr_all_time:.4f}", f"{dt_all_time:.4f}"],
    'Top 20 (s)': [f"{lr_sel_time:.4f}", f"{dt_sel_time:.4f}"],
    'Speedup': [f"{lr_all_time/lr_sel_time:.2f}x", f"{dt_all_time/dt_sel_time:.2f}x"],
})

print(timing_comparison.to_string(index=False))
print(f"\nComputational benefit: 3-4x faster training with selected features")


COMPUTATIONAL EFFICIENCY COMPARISON - Training Time Analysis

              Model All Features (119) - Time (s) Selected Features (20) - Time (s) Time Reduction Speedup Factor
Logistic Regression                        1.4372                            0.3238          77.5%          4.44x
      Decision Tree                        0.6529                            0.1783          72.7%          3.66x


✓ COMPUTATIONAL BENEFITS:

  Logistic Regression:
    - All features time: 1.4372 seconds
    - Selected features time: 0.3238 seconds
    - Time saved: 1113.46 ms

  Decision Tree:
    - All features time: 0.6529 seconds
    - Selected features time: 0.1783 seconds
    - Time saved: 474.62 ms



## Summary: Feature Selection Experiment Conclusion

This experiment conclusively demonstrates that **SelectKBest with k=20 is the optimal choice** for the Network Intrusion Detection model, providing:
- **83.2% dimensionality reduction** (from 119 to 20 features)
- **Minimal accuracy loss** (<1% for Decision Tree, ~2.4% for Logistic Regression)
- **Significant computational speedup** (3.66x faster for Decision Tree, 4.44x faster for Logistic Regression)
- **Excellent practical performance** with >98% accuracy maintained for both models

**Recommendation**: Use SelectKBest (k=20) for production deployment as it offers the best balance between model simplicity, computational efficiency, and predictive accuracy.

## Train Logistic Regression Model

In [47]:
print("Starting Model Training Pipeline")


print("\nTraining Logistic Regression model...")
lr_model = train_logistic_regression(X_train, y_train)
print("Training completed successfully!")

Starting Model Training Pipeline

Training Logistic Regression model...
Training completed successfully!


## Evaluate Logistic Regression Model

In [48]:
print("\nGenerating evaluation metrics...")
evaluate_model(lr_model, X_test, y_test, "Logistic Regression")


Generating evaluation metrics...

=== Logistic Regression ===
Accuracy:  97.14%
Precision: 97.73%
Recall:    96.11%
F1-Score:  96.91%
Confusion Matrix:
[[13159   263]
 [  458 11315]]
------------------------------


## Train Decision Tree Model

In [49]:

print("Training Decision Tree model...")
dt_model = train_decision_tree(X_train, y_train)
print("Training completed successfully!")

Training Decision Tree model...
Training completed successfully!


## Evaluate Decision Tree Model

In [ ]:
print("\nGenerating evaluation metrics")
evaluate_model(dt_model, X_test, y_test, "Decision Tree")

print("\nPipeline execution completed!")


Generating evaluation metrics...

=== Decision Tree ===
Accuracy:  99.66%
Precision: 99.68%
Recall:    99.59%
F1-Score:  99.63%
Confusion Matrix:
[[13384    38]
 [   48 11725]]
------------------------------

Pipeline execution completed!
